# FFNN Model - Multiple Training Runs
This notebook trains the model N times and reports aggregate MAPE statistics

In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import os
import sys
sys.path.append("..")


from utils import load_and_prepare_data, get_error_df
root = os.path.join(os.path.dirname(os.path.abspath(__file__)), '..', 'Data')
denoised_sales_df = pd.read_excel(os.path.join(root, 'FixedData_Random_Smoothed.xlsx'))
noised_sales_df = pd.read_excel(os.path.join(root, 'DataMissingValuesFilled.xlsx'))
holiday_df = pd.read_csv(os.path.join(root, 'holidays.csv'))
crude_df = pd.read_csv(os.path.join(root, 'Crude Oil Prices.csv'))


val_split_date = '2023-06-01'
test_split_date = '2024-01-01'
seq_length = 30
forecast_length = 30

epochs = 150

shift_crude_days = 0
warmup_days = 14
warmup_epochs = 2
warmup_every_n_days = 1

act_fn = 'relu'
loss_fn = 'asymmetric_mse'


In [ ]:
# Place the contents of best_hyperparameters_weighted.txt here.
hyperparameters = {
    "sales_and_crude_no_scaled_calender_numbers": {
        "rank_1": {
            "rank": 1,
            "composite_score": 0.092362,
            "MAPE": 7.1200571567645845,
            "MUPE": 4.673243917944492,
            "MOPE": 8.196517199640693,
            "num_layers": 3,
            "hidden_layers": [
                1216,
                1856,
                1280
            ],
            "dropout": 0.0501705835963308,
            "lr": 0.0004362998740752,
            "batch_size": 32,
            "under_parameter": 0.8127018148777372,
            "over_parameter": 1.4633234947656502
        },
        "rank_2": {
            "rank": 2,
            "composite_score": 0.092546,
            "MAPE": 7.229765765654593,
            "MUPE": 4.4019703304966065,
            "MOPE": 8.268879419490633,
            "num_layers": 3,
            "hidden_layers": [
                1728,
                768,
                1920
            ],
            "dropout": 0.0501705835963308,
            "lr": 0.0004362998740752,
            "batch_size": 32,
            "under_parameter": 1.226964342665685,
            "over_parameter": 1.8261477726433637
        },
        "rank_3": {
            "rank": 3,
            "composite_score": 0.092693,
            "MAPE": 7.1775795420782655,
            "MUPE": 4.678885008773247,
            "MOPE": 8.132663802360392,
            "num_layers": 1,
            "hidden_layers": [
                1216
            ],
            "dropout": 0.1656753829496143,
            "lr": 0.0005237716487688,
            "batch_size": 64,
            "under_parameter": 0.1036821236168126,
            "over_parameter": 0.1412133211968695
        },
        "rank_4": {
            "rank": 4,
            "composite_score": 0.092909,
            "MAPE": 7.10627352842516,
            "MUPE": 4.911361798113366,
            "MOPE": 8.079561377035754,
            "num_layers": 1,
            "hidden_layers": [
                1216
            ],
            "dropout": 0.0501705835963308,
            "lr": 0.0004362998740752,
            "batch_size": 16,
            "under_parameter": 0.8127018148777372,
            "over_parameter": 1.4633234947656502
        },
        "rank_5": {
            "rank": 5,
            "composite_score": 0.092979,
            "MAPE": 7.248954898930084,
            "MUPE": 4.430484679824819,
            "MOPE": 8.266280116812505,
            "num_layers": 1,
            "hidden_layers": [
                1664
            ],
            "dropout": 0.1665137145859003,
            "lr": 0.0002616522116127,
            "batch_size": 64,
            "under_parameter": 1.4021830540312463,
            "over_parameter": 1.394151911045336
        }
    },
    "sales_and_crude_no_scaled_calender_sincos": {
        "rank_1": {
            "rank": 1,
            "composite_score": 0.086926,
            "MAPE": 7.133529468442961,
            "MUPE": 4.8370182851207835,
            "MOPE": 8.123323722051108,
            "num_layers": 2,
            "hidden_layers": [
                2048,
                512
            ],
            "dropout": 0.1267836646689794,
            "lr": 0.0003604466196269,
            "batch_size": 32,
            "under_parameter": 1.4332710825362844,
            "over_parameter": 1.7907557756669534
        },
        "rank_2": {
            "rank": 2,
            "composite_score": 0.087432,
            "MAPE": 7.243428316068852,
            "MUPE": 4.445527603004002,
            "MOPE": 8.289611378615842,
            "num_layers": 2,
            "hidden_layers": [
                1216,
                1536
            ],
            "dropout": 0.1398343737300125,
            "lr": 0.0002863803839853,
            "batch_size": 32,
            "under_parameter": 1.0892590130584892,
            "over_parameter": 1.3654555092687608
        },
        "rank_3": {
            "rank": 3,
            "composite_score": 0.087859,
            "MAPE": 7.2787993532218,
            "MUPE": 4.341960868853898,
            "MOPE": 8.352039570928918,
            "num_layers": 2,
            "hidden_layers": [
                2048,
                1344
            ],
            "dropout": 0.2268520343726582,
            "lr": 0.0004161689705303,
            "batch_size": 32,
            "under_parameter": 1.0892590130584892,
            "over_parameter": 0.9060820639504265
        },
        "rank_4": {
            "rank": 4,
            "composite_score": 0.088678,
            "MAPE": 7.371103517799205,
            "MUPE": 3.935781386637618,
            "MOPE": 8.581887622639409,
            "num_layers": 2,
            "hidden_layers": [
                1600,
                1216
            ],
            "dropout": 0.2268520343726582,
            "lr": 0.0008666682900708,
            "batch_size": 32,
            "under_parameter": 1.0892590130584892,
            "over_parameter": 0.9060820639504265
        },
        "rank_5": {
            "rank": 5,
            "composite_score": 0.089168,
            "MAPE": 7.2347880173936865,
            "MUPE": 4.791933801804546,
            "MOPE": 8.221606582862632,
            "num_layers": 2,
            "hidden_layers": [
                768,
                448
            ],
            "dropout": 0.1093109009447482,
            "lr": 0.0001571880920755,
            "batch_size": 16,
            "under_parameter": 1.4481708898062546,
            "over_parameter": 1.808842142555033
        }
    },
    "sales_and_crude_no_scaled_no_calender": {
        "rank_1": {
            "rank": 1,
            "composite_score": 0.097589,
            "MAPE": 7.328107923427738,
            "MUPE": 3.942740599282287,
            "MOPE": 8.440305310114768,
            "num_layers": 3,
            "hidden_layers": [
                768,
                1600,
                576
            ],
            "dropout": 0.0039764517983655,
            "lr": 0.000805243080772,
            "batch_size": 32,
            "under_parameter": 0.1012869629262767,
            "over_parameter": 0.0768418198976048
        },
        "rank_2": {
            "rank": 2,
            "composite_score": 0.098595,
            "MAPE": 7.161198666884021,
            "MUPE": 5.003505452654303,
            "MOPE": 7.923854441414552,
            "num_layers": 2,
            "hidden_layers": [
                640,
                448
            ],
            "dropout": 0.0307210743680001,
            "lr": 0.0011549158437235,
            "batch_size": 64,
            "under_parameter": 0.8125706611124621,
            "over_parameter": 1.2776283361186225
        },
        "rank_3": {
            "rank": 3,
            "composite_score": 0.098947,
            "MAPE": 7.187170149977863,
            "MUPE": 4.837823873196094,
            "MOPE": 8.05991924295487,
            "num_layers": 2,
            "hidden_layers": [
                768,
                1600
            ],
            "dropout": 0.0039764517983655,
            "lr": 0.0011549158437235,
            "batch_size": 64,
            "under_parameter": 0.8125706611124621,
            "over_parameter": 1.5436524609520803
        },
        "rank_4": {
            "rank": 4,
            "composite_score": 0.098986,
            "MAPE": 7.167049925677672,
            "MUPE": 4.857566734881398,
            "MOPE": 8.081380742085699,
            "num_layers": 2,
            "hidden_layers": [
                640,
                1856
            ],
            "dropout": 0.0307672663494773,
            "lr": 0.0011549158437235,
            "batch_size": 64,
            "under_parameter": 0.8125706611124621,
            "over_parameter": 1.2776283361186225
        },
        "rank_5": {
            "rank": 5,
            "composite_score": 0.099622,
            "MAPE": 7.243496833118321,
            "MUPE": 4.6586958253693025,
            "MOPE": 8.19086480164467,
            "num_layers": 2,
            "hidden_layers": [
                2048,
                384
            ],
            "dropout": 0.0307210743680001,
            "lr": 0.0011549158437235,
            "batch_size": 64,
            "under_parameter": 0.8125706611124621,
            "over_parameter": 1.2776283361186225
        }
    },
    "sales_and_crude_scaled_calender_numbers": {
        "rank_1": {
            "rank": 1,
            "composite_score": 0.040039,
            "MAPE": 6.8567086388670315,
            "MUPE": 4.210721202380387,
            "MOPE": 8.095497397576263,
            "num_layers": 1,
            "hidden_layers": [
                192
            ],
            "dropout": 0.0827585919076395,
            "lr": 0.0025071473459037,
            "batch_size": 16,
            "under_parameter": 0.740048980834549,
            "over_parameter": 0.6352757278917216
        },
        "rank_2": {
            "rank": 2,
            "composite_score": 0.040066,
            "MAPE": 6.8725993966576695,
            "MUPE": 4.329090350254063,
            "MOPE": 7.934378578811023,
            "num_layers": 1,
            "hidden_layers": [
                832
            ],
            "dropout": 0.0827585919076395,
            "lr": 0.0030871974668967,
            "batch_size": 16,
            "under_parameter": 1.6457204284341302,
            "over_parameter": 1.354630805775041
        },
        "rank_3": {
            "rank": 3,
            "composite_score": 0.040113,
            "MAPE": 6.818886907387753,
            "MUPE": 4.444654877070887,
            "MOPE": 7.934408015053994,
            "num_layers": 1,
            "hidden_layers": [
                1664
            ],
            "dropout": 0.2772982879783262,
            "lr": 0.0023099793681305,
            "batch_size": 32,
            "under_parameter": 1.4270348067396796,
            "over_parameter": 1.97903050189685
        },
        "rank_4": {
            "rank": 4,
            "composite_score": 0.042269,
            "MAPE": 6.825121697018058,
            "MUPE": 4.8709564863230215,
            "MOPE": 7.67126332120615,
            "num_layers": 1,
            "hidden_layers": [
                192
            ],
            "dropout": 0.0827585919076395,
            "lr": 0.000932555350527,
            "batch_size": 32,
            "under_parameter": 1.4161205481212318,
            "over_parameter": 1.97903050189685
        },
        "rank_5": {
            "rank": 5,
            "composite_score": 0.048384,
            "MAPE": 7.1468422273913275,
            "MUPE": 4.1324141712139095,
            "MOPE": 8.350756580262615,
            "num_layers": 1,
            "hidden_layers": [
                1152
            ],
            "dropout": 0.2772982879783262,
            "lr": 0.0023099793681305,
            "batch_size": 32,
            "under_parameter": 1.9649485041481272,
            "over_parameter": 1.97903050189685
        }
    },
    "sales_and_crude_scaled_calender_sincos": {
        "rank_1": {
            "rank": 1,
            "composite_score": 0.139775,
            "MAPE": 7.946089707164821,
            "MUPE": 3.889580069407443,
            "MOPE": 9.280338547806464,
            "num_layers": 1,
            "hidden_layers": [
                384
            ],
            "dropout": 0.3223441655234559,
            "lr": 0.0043924196094056,
            "batch_size": 32,
            "under_parameter": 0.5843589999554439,
            "over_parameter": 0.6127657095714361
        },
        "rank_2": {
            "rank": 2,
            "composite_score": 0.147524,
            "MAPE": 7.679143792891247,
            "MUPE": 4.32384993660693,
            "MOPE": 9.081129980329882,
            "num_layers": 1,
            "hidden_layers": [
                384
            ],
            "dropout": 0.252430988875073,
            "lr": 0.0006207910981319,
            "batch_size": 16,
            "under_parameter": 1.8045743956696787,
            "over_parameter": 1.3660492396080222
        },
        "rank_3": {
            "rank": 3,
            "composite_score": 0.154846,
            "MAPE": 8.636064204012587,
            "MUPE": 3.707216008803193,
            "MOPE": 10.017595132856906,
            "num_layers": 1,
            "hidden_layers": [
                640
            ],
            "dropout": 0.4896911706944568,
            "lr": 0.0004993976881946,
            "batch_size": 16,
            "under_parameter": 0.9651248642570924,
            "over_parameter": 0.5189457170447545
        },
        "rank_4": {
            "rank": 4,
            "composite_score": 0.160016,
            "MAPE": 9.3672212757812,
            "MUPE": 3.3106280264769583,
            "MOPE": 10.391459967743282,
            "num_layers": 1,
            "hidden_layers": [
                384
            ],
            "dropout": 0.3223441655234559,
            "lr": 0.0043924196094056,
            "batch_size": 32,
            "under_parameter": 0.4621166731615991,
            "over_parameter": 0.2055848673639553
        },
        "rank_5": {
            "rank": 5,
            "composite_score": 0.162799,
            "MAPE": 9.786087929970344,
            "MUPE": 3.023252669342262,
            "MOPE": 10.82303329202048,
            "num_layers": 1,
            "hidden_layers": [
                1152
            ],
            "dropout": 0.4018479895616418,
            "lr": 0.0008243989819973,
            "batch_size": 32,
            "under_parameter": 0.7482369605261414,
            "over_parameter": 0.3383109777423985
        }
    },
    "sales_and_crude_scaled_no_calender": {
        "rank_1": {
            "rank": 1,
            "composite_score": 0.05593,
            "MAPE": 7.206637404644924,
            "MUPE": 4.213333000004364,
            "MOPE": 8.389712869298176,
            "num_layers": 1,
            "hidden_layers": [
                1664
            ],
            "dropout": 0.1899917726988201,
            "lr": 0.0001809628156339,
            "batch_size": 32,
            "under_parameter": 1.2981910365444516,
            "over_parameter": 1.9302318565341792
        },
        "rank_2": {
            "rank": 2,
            "composite_score": 0.058433,
            "MAPE": 7.059073087008076,
            "MUPE": 4.769368717439457,
            "MOPE": 8.357094724077289,
            "num_layers": 1,
            "hidden_layers": [
                896
            ],
            "dropout": 0.12834760488209,
            "lr": 0.0021758052996314,
            "batch_size": 16,
            "under_parameter": 1.082600672627647,
            "over_parameter": 1.4788950146546609
        },
        "rank_3": {
            "rank": 3,
            "composite_score": 0.059598,
            "MAPE": 7.1411919323549835,
            "MUPE": 4.505651977571844,
            "MOPE": 8.57663947974846,
            "num_layers": 1,
            "hidden_layers": [
                576
            ],
            "dropout": 0.1899917726988201,
            "lr": 0.001049765545732,
            "batch_size": 32,
            "under_parameter": 1.4333488163348302,
            "over_parameter": 1.9177151053337809
        },
        "rank_4": {
            "rank": 4,
            "composite_score": 0.063396,
            "MAPE": 7.411810633823471,
            "MUPE": 4.018329271204579,
            "MOPE": 8.858374272619821,
            "num_layers": 2,
            "hidden_layers": [
                1664,
                576
            ],
            "dropout": 0.1899917726988201,
            "lr": 0.001049765545732,
            "batch_size": 64,
            "under_parameter": 1.6050454271968102,
            "over_parameter": 1.9177151053337809
        },
        "rank_5": {
            "rank": 5,
            "composite_score": 0.066018,
            "MAPE": 7.554896233616892,
            "MUPE": 3.799712881259425,
            "MOPE": 9.023441751256422,
            "num_layers": 1,
            "hidden_layers": [
                960
            ],
            "dropout": 0.0537648474412281,
            "lr": 0.0002853586364109,
            "batch_size": 16,
            "under_parameter": 1.4333488163348302,
            "over_parameter": 1.0644425567496856
        }
    },
    "sales_only_no_scaled_calender_numbers": {
        "rank_1": {
            "rank": 1,
            "composite_score": 0.075567,
            "MAPE": 7.054997555610734,
            "MUPE": 4.55210137293023,
            "MOPE": 8.101336204010614,
            "num_layers": 4,
            "hidden_layers": [
                1792,
                1152,
                640,
                1856
            ],
            "dropout": 0.0424453688518928,
            "lr": 0.0001749572295088,
            "batch_size": 16,
            "under_parameter": 0.7352530265401138,
            "over_parameter": 1.179786577566949
        },
        "rank_2": {
            "rank": 2,
            "composite_score": 0.077681,
            "MAPE": 7.083270005647008,
            "MUPE": 5.07776308843449,
            "MOPE": 7.862231164759044,
            "num_layers": 4,
            "hidden_layers": [
                1792,
                896,
                1664,
                1280
            ],
            "dropout": 0.0210871144439024,
            "lr": 0.0001749572295088,
            "batch_size": 16,
            "under_parameter": 0.735964713449669,
            "over_parameter": 1.345217777667436
        },
        "rank_3": {
            "rank": 3,
            "composite_score": 0.078372,
            "MAPE": 7.1145299059141776,
            "MUPE": 4.985212615730893,
            "MOPE": 7.969104117992604,
            "num_layers": 3,
            "hidden_layers": [
                1088,
                1280,
                704
            ],
            "dropout": 0.0424453688518928,
            "lr": 0.0004944480523872,
            "batch_size": 16,
            "under_parameter": 0.9487084752815704,
            "over_parameter": 1.6779945155817584
        },
        "rank_4": {
            "rank": 4,
            "composite_score": 0.078769,
            "MAPE": 7.088533858095006,
            "MUPE": 4.968485843905461,
            "MOPE": 8.073027708552436,
            "num_layers": 4,
            "hidden_layers": [
                1216,
                448,
                1920,
                448
            ],
            "dropout": 0.0210871144439024,
            "lr": 0.0003239765271333,
            "batch_size": 16,
            "under_parameter": 0.735964713449669,
            "over_parameter": 1.345217777667436
        },
        "rank_5": {
            "rank": 5,
            "composite_score": 0.078997,
            "MAPE": 7.137557699606628,
            "MUPE": 4.960178193745781,
            "MOPE": 8.025320876658641,
            "num_layers": 3,
            "hidden_layers": [
                1792,
                640,
                1024
            ],
            "dropout": 0.0424453688518928,
            "lr": 0.0001749572295088,
            "batch_size": 32,
            "under_parameter": 0.7352530265401138,
            "over_parameter": 1.179786577566949
        }
    },
    "sales_only_no_scaled_calender_sincos": {
        "rank_1": {
            "rank": 1,
            "composite_score": 0.09269,
            "MAPE": 7.119169821496937,
            "MUPE": 4.861287560576721,
            "MOPE": 7.979112749761441,
            "num_layers": 1,
            "hidden_layers": [
                1408
            ],
            "dropout": 0.1149131838449758,
            "lr": 0.0002139865794569,
            "batch_size": 16,
            "under_parameter": 0.6119447603223966,
            "over_parameter": 1.1673264690514082
        },
        "rank_2": {
            "rank": 2,
            "composite_score": 0.094384,
            "MAPE": 7.254328453839832,
            "MUPE": 4.376883007042254,
            "MOPE": 8.351143594518199,
            "num_layers": 2,
            "hidden_layers": [
                1344,
                1728
            ],
            "dropout": 0.1977248241401527,
            "lr": 0.0002325846175936,
            "batch_size": 16,
            "under_parameter": 1.9807041228054705,
            "over_parameter": 1.948622613388513
        },
        "rank_3": {
            "rank": 3,
            "composite_score": 0.09439,
            "MAPE": 7.221427524907914,
            "MUPE": 4.591763210456494,
            "MOPE": 8.224590438178,
            "num_layers": 1,
            "hidden_layers": [
                1280
            ],
            "dropout": 0.1149131838449758,
            "lr": 0.000447415773246,
            "batch_size": 64,
            "under_parameter": 1.497359051059964,
            "over_parameter": 1.948622613388513
        },
        "rank_4": {
            "rank": 4,
            "composite_score": 0.09441,
            "MAPE": 7.265754589917437,
            "MUPE": 4.444614800790757,
            "MOPE": 8.276519062732424,
            "num_layers": 2,
            "hidden_layers": [
                2048,
                832
            ],
            "dropout": 0.1977248241401527,
            "lr": 0.0002325846175936,
            "batch_size": 64,
            "under_parameter": 1.4535913481332037,
            "over_parameter": 1.7626419354996044
        },
        "rank_5": {
            "rank": 5,
            "composite_score": 0.094566,
            "MAPE": 7.223536235210718,
            "MUPE": 4.5720234338352475,
            "MOPE": 8.257678204398266,
            "num_layers": 1,
            "hidden_layers": [
                1216
            ],
            "dropout": 0.2745435169684901,
            "lr": 0.0002139865794569,
            "batch_size": 16,
            "under_parameter": 0.6119447603223966,
            "over_parameter": 0.7763988433604745
        }
    },
    "sales_only_no_scaled_no_calender": {
        "rank_1": {
            "rank": 1,
            "composite_score": 0.078739,
            "MAPE": 7.22951386889777,
            "MUPE": 4.569975829546327,
            "MOPE": 8.165486307370234,
            "num_layers": 1,
            "hidden_layers": [
                1984
            ],
            "dropout": 0.0899275207857202,
            "lr": 0.0001641764739287,
            "batch_size": 64,
            "under_parameter": 0.493774672796704,
            "over_parameter": 0.4705149144590241
        },
        "rank_2": {
            "rank": 2,
            "composite_score": 0.078776,
            "MAPE": 7.167874728676106,
            "MUPE": 4.805459371549849,
            "MOPE": 8.050140564746561,
            "num_layers": 1,
            "hidden_layers": [
                1984
            ],
            "dropout": 0.01084835339789,
            "lr": 0.0002552861917556,
            "batch_size": 32,
            "under_parameter": 0.9286876801198276,
            "over_parameter": 1.5179963488150332
        },
        "rank_3": {
            "rank": 3,
            "composite_score": 0.078952,
            "MAPE": 7.287086894052942,
            "MUPE": 4.170938904635512,
            "MOPE": 8.495991205524671,
            "num_layers": 1,
            "hidden_layers": [
                1408
            ],
            "dropout": 0.0733135889715642,
            "lr": 0.0001072682950251,
            "batch_size": 16,
            "under_parameter": 1.6614646746978503,
            "over_parameter": 1.5995331771529742
        },
        "rank_4": {
            "rank": 4,
            "composite_score": 0.079288,
            "MAPE": 7.283604382197661,
            "MUPE": 4.2057647599122845,
            "MOPE": 8.516439023155,
            "num_layers": 1,
            "hidden_layers": [
                1728
            ],
            "dropout": 0.1840385179769232,
            "lr": 0.0001072682950251,
            "batch_size": 16,
            "under_parameter": 0.7510907095253054,
            "over_parameter": 1.025694599354506
        },
        "rank_5": {
            "rank": 5,
            "composite_score": 0.079327,
            "MAPE": 7.23644367520364,
            "MUPE": 4.484633761129455,
            "MOPE": 8.327200127968927,
            "num_layers": 1,
            "hidden_layers": [
                1728
            ],
            "dropout": 0.3146026455629108,
            "lr": 0.000491019358065,
            "batch_size": 16,
            "under_parameter": 0.5859221385335467,
            "over_parameter": 0.8301005073790282
        }
    },
    "sales_only_scaled_calender_numbers": {
        "rank_1": {
            "rank": 1,
            "composite_score": 0.14838,
            "MAPE": 7.156574460279769,
            "MUPE": 4.094356481411865,
            "MOPE": 8.56510368094629,
            "num_layers": 1,
            "hidden_layers": [
                960
            ],
            "dropout": 0.1755012641173602,
            "lr": 0.0034579314101794,
            "batch_size": 32,
            "under_parameter": 1.9871082396877235,
            "over_parameter": 1.7852368759895136
        },
        "rank_2": {
            "rank": 2,
            "composite_score": 0.161792,
            "MAPE": 7.432796454165886,
            "MUPE": 4.003002272363241,
            "MOPE": 8.851994506170055,
            "num_layers": 1,
            "hidden_layers": [
                384
            ],
            "dropout": 0.2356385973109435,
            "lr": 0.002080401705973,
            "batch_size": 32,
            "under_parameter": 1.0105631611491674,
            "over_parameter": 0.7944021441293494
        },
        "rank_3": {
            "rank": 3,
            "composite_score": 0.17605,
            "MAPE": 7.710635319429832,
            "MUPE": 3.940019263827782,
            "MOPE": 9.058661759367858,
            "num_layers": 1,
            "hidden_layers": [
                1024
            ],
            "dropout": 0.3684068511857669,
            "lr": 0.002080401705973,
            "batch_size": 16,
            "under_parameter": 1.590171440996844,
            "over_parameter": 0.8475709772259765
        },
        "rank_4": {
            "rank": 4,
            "composite_score": 0.180485,
            "MAPE": 7.198243575012128,
            "MUPE": 4.445250628748967,
            "MOPE": 8.484935318741154,
            "num_layers": 1,
            "hidden_layers": [
                1536
            ],
            "dropout": 0.3656898890394126,
            "lr": 0.0024798938429793,
            "batch_size": 64,
            "under_parameter": 1.6048540730825145,
            "over_parameter": 1.598106140930385
        },
        "rank_5": {
            "rank": 5,
            "composite_score": 0.184978,
            "MAPE": 8.110333591627974,
            "MUPE": 3.72414814076212,
            "MOPE": 9.342640741553478,
            "num_layers": 1,
            "hidden_layers": [
                1536
            ],
            "dropout": 0.4382505969923645,
            "lr": 0.0029589966632446,
            "batch_size": 32,
            "under_parameter": 0.5247817027180541,
            "over_parameter": 0.2622388932180752
        }
    },
    "sales_only_scaled_calender_sincos": {
        "rank_1": {
            "rank": 1,
            "composite_score": 0.053395,
            "MAPE": 7.0841715160219,
            "MUPE": 4.655986409867069,
            "MOPE": 8.641419684154839,
            "num_layers": 1,
            "hidden_layers": [
                320
            ],
            "dropout": 0.4958074333342646,
            "lr": 0.002442965954254,
            "batch_size": 16,
            "under_parameter": 0.2734841956268987,
            "over_parameter": 0.5491187342747725
        },
        "rank_2": {
            "rank": 2,
            "composite_score": 0.080634,
            "MAPE": 8.385320176999702,
            "MUPE": 3.7443658090609064,
            "MOPE": 9.71101354320753,
            "num_layers": 1,
            "hidden_layers": [
                128
            ],
            "dropout": 0.263762594927339,
            "lr": 0.0011376614945844,
            "batch_size": 16,
            "under_parameter": 1.6816296213335182,
            "over_parameter": 0.9869093407898132
        },
        "rank_3": {
            "rank": 3,
            "composite_score": 0.087644,
            "MAPE": 8.257420650375723,
            "MUPE": 4.339470744283289,
            "MOPE": 9.757156941945087,
            "num_layers": 1,
            "hidden_layers": [
                320
            ],
            "dropout": 0.4230938542845062,
            "lr": 0.0011525595426145,
            "batch_size": 64,
            "under_parameter": 1.845618529254297,
            "over_parameter": 1.2089290410214393
        },
        "rank_4": {
            "rank": 4,
            "composite_score": 0.088943,
            "MAPE": 8.356090043705255,
            "MUPE": 4.113571316484376,
            "MOPE": 10.012032657842294,
            "num_layers": 1,
            "hidden_layers": [
                320
            ],
            "dropout": 0.4958074333342646,
            "lr": 0.002442965954254,
            "batch_size": 64,
            "under_parameter": 0.2734841956268987,
            "over_parameter": 0.1661282659799647
        },
        "rank_5": {
            "rank": 5,
            "composite_score": 0.092185,
            "MAPE": 8.248403077703854,
            "MUPE": 4.413803437039781,
            "MOPE": 10.10801780892217,
            "num_layers": 8,
            "hidden_layers": [
                768,
                1088,
                256,
                704,
                448,
                576,
                1216,
                1280
            ],
            "dropout": 0.2604922316237186,
            "lr": 0.0001640138181456,
            "batch_size": 32,
            "under_parameter": 0.3419066045640322,
            "over_parameter": 0.2466502761801803
        }
    },
    "sales_only_scaled_no_calender": {
        "rank_1": {
            "rank": 1,
            "composite_score": 0.082828,
            "MAPE": 7.240682672415852,
            "MUPE": 4.255323051900349,
            "MOPE": 8.404205188284616,
            "num_layers": 1,
            "hidden_layers": [
                1280
            ],
            "dropout": 0.1769884641080014,
            "lr": 0.0002004853847112,
            "batch_size": 32,
            "under_parameter": 1.3144830286077338,
            "over_parameter": 1.4828666750821375
        },
        "rank_2": {
            "rank": 2,
            "composite_score": 0.082844,
            "MAPE": 7.088576807291438,
            "MUPE": 4.778762056132787,
            "MOPE": 8.225351738284312,
            "num_layers": 1,
            "hidden_layers": [
                704
            ],
            "dropout": 0.3033949530363158,
            "lr": 0.0003488477165583,
            "batch_size": 16,
            "under_parameter": 0.9065392261824424,
            "over_parameter": 1.4950881648761036
        },
        "rank_3": {
            "rank": 3,
            "composite_score": 0.083046,
            "MAPE": 7.202477558168222,
            "MUPE": 4.407140750680139,
            "MOPE": 8.364991113711678,
            "num_layers": 1,
            "hidden_layers": [
                704
            ],
            "dropout": 0.3033949530363158,
            "lr": 0.0003488477165583,
            "batch_size": 16,
            "under_parameter": 0.9065392261824424,
            "over_parameter": 1.4950881648761036
        },
        "rank_4": {
            "rank": 4,
            "composite_score": 0.083173,
            "MAPE": 7.1492975795663725,
            "MUPE": 4.572931634670988,
            "MOPE": 8.32996704367935,
            "num_layers": 2,
            "hidden_layers": [
                1088,
                448
            ],
            "dropout": 0.0777010909029639,
            "lr": 0.0009131059878964,
            "batch_size": 32,
            "under_parameter": 1.534262504889442,
            "over_parameter": 1.7576232519438657
        },
        "rank_5": {
            "rank": 5,
            "composite_score": 0.083445,
            "MAPE": 7.232829727442335,
            "MUPE": 4.4009511239007,
            "MOPE": 8.360674453242762,
            "num_layers": 1,
            "hidden_layers": [
                1152
            ],
            "dropout": 0.0250929256132829,
            "lr": 0.0014939209406754,
            "batch_size": 64,
            "under_parameter": 1.1911026316230482,
            "over_parameter": 1.4828666750821375
        }
    }
}


In [ ]:
from itertools import product

N_RUNS = 20

EXPERIMENTS = [
    # -------------------- SALES ONLY --------------------
    {
        "name": "sales_only_no_scaled_no_calender",
        "use_scaling": False,
        "use_crude": False,
        "use_month": False,
        "use_month_sin_cos": False,
        "use_dayofweek": False,
        "use_dayofweek_sin_cos": False,
    },
    {
        "name": "sales_only_no_scaled_calender_numbers",
        "use_scaling": False,
        "use_crude": False,
        "use_month": True,
        "use_month_sin_cos": False,
        "use_dayofweek": True,
        "use_dayofweek_sin_cos": False,
    },
    {
        "name": "sales_only_no_scaled_calender_sincos",
        "use_scaling": False,
        "use_crude": False,
        "use_month": False,
        "use_month_sin_cos": True,
        "use_dayofweek": False,
        "use_dayofweek_sin_cos": True,
    },
    {
        "name": "sales_only_scaled_no_calender",
        "use_scaling": True,
        "use_crude": False,
        "use_month": False,
        "use_month_sin_cos": False,
        "use_dayofweek": False,
        "use_dayofweek_sin_cos": False,
    },
    {
        "name": "sales_only_scaled_calender_numbers",
        "use_scaling": True,
        "use_crude": False,
        "use_month": True,
        "use_month_sin_cos": False,
        "use_dayofweek": True,
        "use_dayofweek_sin_cos": False,
    },
    {
        "name": "sales_only_scaled_calender_sincos",
        "use_scaling": True,
        "use_crude": False,
        "use_month": False,
        "use_month_sin_cos": True,
        "use_dayofweek": False,
        "use_dayofweek_sin_cos": True,
    },

    # -------------------- SALES + CRUDE --------------------
    {
        "name": "sales_and_crude_no_scaled_no_calender",
        "use_scaling": False,
        "use_crude": True,
        "use_month": False,
        "use_month_sin_cos": False,
        "use_dayofweek": False,
        "use_dayofweek_sin_cos": False,
    },
    {
        "name": "sales_and_crude_no_scaled_calender_numbers",
        "use_scaling": False,
        "use_crude": True,
        "use_month": True,
        "use_month_sin_cos": False,
        "use_dayofweek": True,
        "use_dayofweek_sin_cos": False,
    },
    {
        "name": "sales_and_crude_no_scaled_calender_sincos",
        "use_scaling": False,
        "use_crude": True,
        "use_month": False,
        "use_month_sin_cos": True,
        "use_dayofweek": False,
        "use_dayofweek_sin_cos": True,
    },
    {
        "name": "sales_and_crude_scaled_no_calender",
        "use_scaling": True,
        "use_crude": True,
        "use_month": False,
        "use_month_sin_cos": False,
        "use_dayofweek": False,
        "use_dayofweek_sin_cos": False,
    },
    {
        "name": "sales_and_crude_scaled_calender_numbers",
        "use_scaling": True,
        "use_crude": True,
        "use_month": True,
        "use_month_sin_cos": False,
        "use_dayofweek": True,
        "use_dayofweek_sin_cos": False,
    },
    {
        "name": "sales_and_crude_scaled_calender_sincos",
        "use_scaling": True,
        "use_crude": True,
        "use_month": False,
        "use_month_sin_cos": True,
        "use_dayofweek": False,
        "use_dayofweek_sin_cos": True,
    },
]


In [ ]:
def build_datasets_for_config(cfg):
  sales_scaler = MinMaxScaler() if cfg["use_scaling"] else None
  crude_scaler = MinMaxScaler() if cfg["use_scaling"] else None

  train_dataset, val_dataset, test_dataset = load_and_prepare_data(
      denoised_sales_df=denoised_sales_df,
      noised_sales_df=noised_sales_df,
      crude_df=crude_df,
      val_split_date=val_split_date,
      test_split_date=test_split_date,
      seq_length=seq_length,
      forecast_length=forecast_length,
      batch_size=batch_size,
      shift_crude_days=shift_crude_days,
      use_crude=cfg["use_crude"],
      use_month=cfg["use_month"],
      use_month_sin_cos=cfg["use_month_sin_cos"],
      use_dayofweek=cfg["use_dayofweek"],
      use_dayofweek_sin_cos=cfg["use_dayofweek_sin_cos"],
      sales_scaler=sales_scaler,
      crude_scaler=crude_scaler
  )

  return train_dataset, val_dataset, test_dataset


In [ ]:
from engine import train_model, generate_predictions, generate_warm_up_predictions
from holidayCorrection import holiday_correction
from Models import FFNN
import json

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}\n")

for cfg in EXPERIMENTS:
    print(f"\n{'#'*70}")
    print(f"Experiment: {cfg['name']}")
    print(f"{'#'*70}\n")

    cfg_results = {}

    for rank_key in [f"rank_{i}" for i in range(1, 6)]:
        hp = hyperparameters[cfg['name']][rank_key]

        num_layers       = hp['num_layers']
        hidden_layers    = hp['hidden_layers']
        dropout          = hp['dropout']
        lr               = hp['lr']
        batch_size       = hp['batch_size']
        under_parameter  = hp['under_parameter']
        over_parameter   = hp['over_parameter']

        train_dataset, val_dataset, test_dataset = build_datasets_for_config(cfg)

        all_corrected_mapes       = []
        all_over_prediction_errors  = []
        all_under_prediction_errors = []

        for run_idx in range(N_RUNS):
            print(f"\n{'='*60}")
            print(f"[{cfg['name']}] [{rank_key}] Run {run_idx + 1}/{N_RUNS}")
            print(f"{'='*60}")

            input_dim = train_dataset.X.shape[1] * train_dataset.X.shape[2]

            model = FFNN(
                input_size=input_dim,
                output_size=forecast_length,
                hidden_layers=hidden_layers,
                activation=act_fn,
                dropout=dropout
            ).to(device)

            if run_idx == 0:
                print(f"Model Parameters: {sum(p.numel() for p in model.parameters()):,}\n")

            train_loader = train_dataset.get_dataloader(batch_size=batch_size, shuffle=True)
            val_loader   = val_dataset.get_dataloader(batch_size=batch_size, shuffle=False)

            train_losses, test_losses, optimizer, criterion = train_model(
                model=model,
                train_loader=train_loader,
                val_loader=val_loader,
                epochs=epochs,
                lr=lr,
                device=device,
                under_parameter=under_parameter,
                over_parameter=over_parameter,
                patience=50,
                loss_fn=loss_fn
            )

            print("Performing Predictions")
            Y_pred_uncorrected, Y_true, Y_true_noised = generate_predictions(
                model, test_dataset, device=device
            )
            Y_pred_corrected = holiday_correction(Y_pred_uncorrected, test_dataset.sample_dates, noised_sales_df, holiday_df)

            error_df = get_error_df(
                Y_true_noised=Y_true_noised,
                Y_pred_corrected=Y_pred_corrected,
                Y_pred_uncorrected=Y_pred_uncorrected,
                sample_dates=test_dataset.sample_dates
            )

            corrected_mape        = error_df['CorrectedMape'].mean()
            over_prediction_error = error_df['ErrorOverPrediction'].mean()
            under_prediction_error = error_df['ErrorUnderPrediction'].mean()

            all_corrected_mapes.append(corrected_mape)
            all_over_prediction_errors.append(over_prediction_error)
            all_under_prediction_errors.append(under_prediction_error)

            print(f"Run {run_idx+1} | MAPE: {corrected_mape:.3f}")

        cfg_results[rank_key] = {
            "mape_mean":  np.mean(all_corrected_mapes),
            "mape_std":   np.std(all_corrected_mapes),
            "over_mean":  np.mean(all_over_prediction_errors),
            "over_std":   np.std(all_over_prediction_errors),
            "under_mean": np.mean(all_under_prediction_errors),
            "under_std":  np.std(all_under_prediction_errors)
        }

        print(f"\n✔ [{cfg['name']}] [{rank_key}] completed")
        print(f"MAPE: {cfg_results[rank_key]['mape_mean']:.3f} ± {cfg_results[rank_key]['mape_std']:.3f}")

    # ── per-config summary ──────────────────────────────────────────────
    print(f"\n\n{'='*60}")
    print(f"AGGREGATE RESULTS FOR {cfg['name']} ACROSS {N_RUNS} RUNS")
    print(f"{'='*60}\n")

    for rank_key, res in cfg_results.items():
        print(f"{rank_key}")
        print(f"MAPE: {res['mape_mean']:.3f} ± {res['mape_std']:.3f}")
        print(f"Over Prediction Error:  {res['over_mean']:.3f} ± {res['over_std']:.3f}")
        print(f"Under Prediction Error: {res['under_mean']:.3f} ± {res['under_std']:.3f}")
        print()

    # ── save to separate file per config ───────────────────────────────
    with open(f"Results_{cfg['name']}.txt", 'w') as f:
        json.dump(cfg_results, f, indent=4)